# ReadyNow! — Case Study 6
### Federal Emergency Machine Assistant · Agentic AI with Google ADK

**Architecture**

```
                          USER
                            |
              [before_model_callback]  <-- logs every prompt + validates input
                            |            (US-location check, injection check, scope check)
                            v
                 ReadyNow_Root  (Agent)
                  "describes capabilities, coordinates"
                            |  sub_agents  (ADK transfer)
                            v
        ReadyNow_ResponseTeam  (SequentialAgent)
        +-------------------+-------------------+
        |                   |                   |
        v                   v                   v
   1. Responder   -->  2. Critic     -->   3. Refiner
   (drafts answer)     (validates)         (rewrites)
   output_key:          output_key:         output_key:
   draft_answer         critique            final_briefing
        |
        |  tools = AgentTool(...)   <-- control RETURNS to responder
        +---------------+---------------+
        |               |               |
        v               v               v
   Weather Agent   Search Agent   Routes Agent
   - get_lat_lon   - google_search  - get_evacuation_route
   - get_nws_alerts  (ADK built-in)   (Google Maps
   - get_nws_forecast                  Directions API)
     (NWS API)
                            |
              [after_model_callback]  <-- logs every model response
                            v
                  Vertex AI Agent Engine
```

**Rubric coverage**

| Requirement | Where |
|---|---|
| Root agent describing capabilities, coordinating sub-agents | `readynow_root` (Cell 1) |
| Weather / search / Maps routes / Q&A agents | `weather_agent`, `search_agent`, `routes_agent`, `responder` |
| Sequential workflow that validates and refines | `ReadyNow_ResponseTeam` (SequentialAgent) |
| Callbacks logging all user–agent interactions | `log_and_validate_prompt`, `log_model_response` |
| User input validation | same before-model callback |
| Deploy to Agent Platform | Cell 3 |
| Test code | Cells 2 and 3 |




# Environment setup and deployment

In [1]:
# =============================================================================
# SETUP & ARCHITECTURE
# =============================================================================
# %pip install -q "google-cloud-aiplatform[adk,agent_engines]" google-adk requests

import logging
import os
import re
from typing import Any, Dict

import requests
import vertexai
from google.adk.agents import Agent, SequentialAgent
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmRequest, LlmResponse
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool
from google.cloud import storage
from google.genai import types
from vertexai import agent_engines

# ----------------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------------
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "")  # auto-set in Colab Enterprise
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")
BUCKET_NAME = f"{PROJECT_ID}-readynow-staging"
STAGING_BUCKET = f"gs://{BUCKET_NAME}"
DISPLAY_NAME = "readynow-emergency-agent"
MODEL = "gemini-2.5-flash"

# NOTE: Directions/Geocoding run in fallback mode. The Maps code path in get_lat_lon and
# get_evacuation_route is complete and activates when GOOGLE_MAPS_API_KEY is set.
os.environ.setdefault("GOOGLE_MAPS_API_KEY", "")

# NWS asks for a User-Agent that identifies your app and gives a REAL contact address.
os.environ.setdefault("NWS_USER_AGENT", "ReadyNowWorkshop/1.0 (your.email@example.com)")

assert PROJECT_ID, "Set PROJECT_ID — GOOGLE_CLOUD_PROJECT was not found in the environment."

vertexai.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET)

# Create the staging bucket if it does not exist (vertexai.init does NOT create it).
_sc = storage.Client(project=PROJECT_ID)
if _sc.lookup_bucket(BUCKET_NAME) is None:
    _sc.create_bucket(BUCKET_NAME, location=LOCATION)
    print(f"Created staging bucket {STAGING_BUCKET}")

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("ReadyNow")

NWS_PRECISION = 4  # api.weather.gov rejects coordinates with >4 decimal places
HTTP_TIMEOUT = 10


# =============================================================================
# 1. TOOLS
#
# Every tool returns a dict with an explicit "status" key: for a
# life-safety agent the model MUST be able to tell "no alerts" apart from "the
# lookup failed", and must never fall back to a guessed location.
# =============================================================================


def get_lat_lon(location_name: str) -> Dict[str, Any]:
    """Resolve a U.S. place name to latitude and longitude.

    Args:
        location_name (str): A U.S. city, address, or ZIP code (e.g. "Bowie, MD").

    Returns:
        Dict[str, Any]: {"status": "ok", "lat": float, "lon": float,
        "resolved_name": str} on success, or {"status": "error", ...} on failure.
        Never guesses a location.
    """
    key = os.environ.get("GOOGLE_MAPS_API_KEY")
    if key:
        try:
            res = requests.get(
                "https://maps.googleapis.com/maps/api/geocode/json",
                params={"address": location_name, "key": key},
                timeout=HTTP_TIMEOUT,
            ).json()
            if res.get("status") == "OK" and res.get("results"):
                top = res["results"][0]
                loc = top["geometry"]["location"]
                return {
                    "status": "ok",
                    "lat": round(loc["lat"], NWS_PRECISION),
                    "lon": round(loc["lng"], NWS_PRECISION),
                    "resolved_name": top.get("formatted_address", location_name),
                }
        except Exception as exc:
            logger.warning("Geocoding (Maps) failed: %s", exc)

    try:
        res = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": location_name, "count": 1, "country": "US", "format": "json"},
            timeout=HTTP_TIMEOUT,
        ).json()
        if res.get("results"):
            loc = res["results"][0]
            return {
                "status": "ok",
                "lat": round(loc["latitude"], NWS_PRECISION),
                "lon": round(loc["longitude"], NWS_PRECISION),
                "resolved_name": loc.get("name", location_name),
            }
    except Exception as exc:
        logger.warning("Geocoding (Open-Meteo) failed: %s", exc)

    return {
        "status": "error",
        "message": f"Could not resolve '{location_name}'. Ask for a city and state or a ZIP code.",
    }


def get_nws_alerts(lat: float, lon: float) -> Dict[str, Any]:
    """Fetch active National Weather Service alerts for a coordinate pair.

    Args:
        lat (float): Latitude in decimal degrees (e.g. 38.9427).
        lon (float): Longitude in decimal degrees (e.g. -76.7305).

    Returns:
        Dict[str, Any]: {"status": "ok", "alert_count": int, "alerts": list} where an
        empty list means no active alerts, or {"status": "error", ...} meaning the
        lookup did not complete and must NOT be reported as an all-clear.
    """
    point = f"{round(lat, NWS_PRECISION)},{round(lon, NWS_PRECISION)}"
    try:
        res = requests.get(
            "https://api.weather.gov/alerts/active",
            params={"point": point},
            headers={"User-Agent": os.environ["NWS_USER_AGENT"], "Accept": "application/geo+json"},
            timeout=HTTP_TIMEOUT,
        )
        if res.status_code != 200:
            return {
                "status": "error",
                "message": f"NWS returned HTTP {res.status_code}. Do NOT say there are no alerts.",
            }
        alerts = [
            {
                "event": f["properties"].get("event"),
                "severity": f["properties"].get("severity"),
                "urgency": f["properties"].get("urgency"),
                "area": f["properties"].get("areaDesc"),
                "headline": f["properties"].get("headline"),
                "instruction": f["properties"].get("instruction"),
            }
            for f in res.json().get("features", [])
        ]
        return {"status": "ok", "point": point, "alert_count": len(alerts), "alerts": alerts}
    except Exception as exc:
        logger.warning("NWS alerts failed: %s", exc)
        return {"status": "error", "message": "Could not reach the NWS. Do NOT say there are no alerts."}


def get_nws_forecast(lat: float, lon: float) -> Dict[str, Any]:
    """Fetch the short-term National Weather Service forecast for a coordinate pair.

    Args:
        lat (float): Latitude in decimal degrees.
        lon (float): Longitude in decimal degrees.

    Returns:
        Dict[str, Any]: {"status": "ok", "periods": list} with the next few forecast
        periods, or {"status": "error", ...}.
    """
    point = f"{round(lat, NWS_PRECISION)},{round(lon, NWS_PRECISION)}"
    headers = {"User-Agent": os.environ["NWS_USER_AGENT"], "Accept": "application/geo+json"}
    try:
        pts = requests.get(f"https://api.weather.gov/points/{point}", headers=headers, timeout=HTTP_TIMEOUT)
        if pts.status_code != 200:
            return {"status": "error", "message": f"NWS points lookup returned HTTP {pts.status_code}."}
        fc = requests.get(pts.json()["properties"]["forecast"], headers=headers, timeout=HTTP_TIMEOUT)
        if fc.status_code != 200:
            return {"status": "error", "message": f"NWS forecast returned HTTP {fc.status_code}."}
        periods = [
            {
                "name": p.get("name"),
                "temperature": f"{p.get('temperature')} {p.get('temperatureUnit')}",
                "wind": f"{p.get('windSpeed')} {p.get('windDirection')}",
                "forecast": p.get("shortForecast"),
            }
            for p in fc.json()["properties"]["periods"][:4]
        ]
        return {"status": "ok", "periods": periods}
    except Exception as exc:
        logger.warning("NWS forecast failed: %s", exc)
        return {"status": "error", "message": "Could not reach the NWS forecast service."}


def get_evacuation_route(origin: str, destination: str) -> Dict[str, Any]:
    """Retrieve a driving route between two places for evacuation planning.

    Args:
        origin (str): Starting address or place name.
        destination (str): Destination address or place name.

    Returns:
        Dict[str, Any]: {"status": "ok", "distance": str, "duration": str,
        "first_steps": list, "advisory": str}, or {"status": "unavailable", ...}
        with general guidance. Routing is advisory and never overrides official orders.
    """
    key = os.environ.get("GOOGLE_MAPS_API_KEY")
    if key:
        try:
            res = requests.get(
                "https://maps.googleapis.com/maps/api/directions/json",
                params={"origin": origin, "destination": destination, "key": key},
                timeout=HTTP_TIMEOUT,
            ).json()
            if res.get("status") == "OK" and res.get("routes"):
                leg = res["routes"][0]["legs"][0]
                # Directions returns HTML markup in html_instructions - strip it.
                steps = [
                    re.sub(r"<[^>]+>", " ", s.get("html_instructions", "")).strip()
                    for s in leg.get("steps", [])[:5]
                ]
                return {
                    "status": "ok",
                    "distance": leg["distance"]["text"],
                    "duration": leg["duration"]["text"],
                    "start_address": leg["start_address"],
                    "end_address": leg["end_address"],
                    "first_steps": [s for s in steps if s],
                    "advisory": "Advisory only. Official evacuation orders and posted traffic control take precedence.",
                }
        except Exception as exc:
            logger.warning("Directions failed: %s", exc)

    return {
        "status": "unavailable",
        "message": f"Could not compute a route from {origin} to {destination}.",
        "advisory": "Follow the route designated by local emergency management. Call 911 if you are trapped.",
    }


# =============================================================================
# 2. CALLBACKS — logging every interaction + validating user input
# =============================================================================

INJECTION_PATTERNS = [
    r"ignore\s+(all\s+)?(previous|prior)\s+instructions",
    r"system\s+override",
    r"drop\s+table",
    r"<\s*script",
]

# Demo-grade heuristic. A real deployment would use Model Armor / Vertex AI
# safety filters -- a four-item regex list is not a serious injection defense.
NON_US_HINTS = [
    "london", "paris", "tokyo", "berlin", "madrid", "toronto", "sydney",
    "mumbai", "lagos", "mexico city", "beijing", "moscow", "dubai", "cairo",
]

# Regex, not substrings: "write an essay" failed to match "write ME an essay",
# and one inserted word should not defeat the filter.
OFF_TOPIC_PATTERNS = [
    r"\brecipe\b",
    r"\bwrite\b.{0,25}\b(essay|poem|story|blog post|article|song)\b",
    r"\b(movie|film|book|tv show|music)\s+recommendation",
    r"\b(tell|say|know)\b.{0,20}\bjoke\b",
    r"\bwho won\b",
    r"\bhomework\b",
]

# Scope enforcement only fires when NO emergency keyword is present. Refusing
# "is this a joke, my street is flooding" would be far worse than answering an
# off-topic question.
EMERGENCY_KEYWORDS = [
    "weather", "storm", "hurricane", "tornado", "flood", "fire", "evacuat", "shelter",
    "alert", "warning", "emergency", "safe", "safety", "disaster", "earthquake",
    "forecast", "route", "911",
]
# "help" was deliberately removed: it appears in almost any phrasing and let
# off-topic requests through the scope check.


def _text_of(content) -> str:
    if not content or not getattr(content, "parts", None):
        return ""
    return " ".join(p.text for p in content.parts if getattr(p, "text", None)).strip()


def _refuse(msg: str) -> LlmResponse:
    return LlmResponse(content=types.Content(role="model", parts=[types.Part(text=msg)]))


def log_and_validate_prompt(callback_context: CallbackContext, llm_request: LlmRequest):
    """before_model_callback: logs the user prompt, then validates it.

    ADK calls this with KEYWORD arguments, so the parameter names
    'callback_context' and 'llm_request' are part of the contract.
    """
    if not llm_request or not llm_request.contents:
        return None

    user_text = ""
    for content in reversed(llm_request.contents):
        if getattr(content, "role", None) == "user":
            user_text = _text_of(content)
            if user_text:
                break
    if not user_text:
        return None

    logger.info("[%s] USER >> %s", callback_context.agent_name, user_text)

    if callback_context.agent_name != "ReadyNow_Root":
        return None

    low = user_text.lower()

    if any(re.search(p, low) for p in INJECTION_PATTERNS):
        logger.warning("[SECURITY] blocked prompt pattern")
        return _refuse("I can only help with weather alerts, disaster safety, evacuation routing, and shelters.")

    if any(city in low for city in NON_US_HINTS):
        return _refuse(
            "ReadyNow! uses the U.S. National Weather Service, which only covers U.S. locations. "
            "Give me a U.S. city and state or ZIP code."
        )

    if any(re.search(p, low) for p in OFF_TOPIC_PATTERNS) and not any(
        k in low for k in EMERGENCY_KEYWORDS
    ):
        return _refuse(
            "I'm FEMA's ReadyNow! assistant — I handle weather alerts, disaster safety, "
            "evacuation routing, and shelter resources. What do you need help with?"
        )

    return None


def log_model_response(callback_context: CallbackContext, llm_response: LlmResponse):
    """after_model_callback: logs the model's response for the audit trail."""
    if llm_response and llm_response.content and llm_response.content.parts:
        txt = llm_response.content.parts[0].text
        if txt:
            logger.info("[%s] MODEL >> %s", callback_context.agent_name, txt.strip()[:400])
    return None


CALLBACKS = {
    "before_model_callback": log_and_validate_prompt,
    "after_model_callback": log_model_response,
}


# =============================================================================
# 3. SPECIALIST AGENTS  (invoked as AgentTools so control returns)
# =============================================================================

weather_agent = Agent(
    name="ReadyNow_Weather",
    model=MODEL,
    description="Reports active NWS severe weather alerts and the short-term forecast for a U.S. location.",
    instruction=(
        "Run get_lat_lon for the requested location, then get_nws_alerts and, if asked for "
        "conditions, get_nws_forecast using the returned coordinates. If any tool returns "
        "status 'error', say so explicitly. Only report 'no active alerts' when a tool "
        "returned status 'ok' with an empty alerts list."
    ),
    tools=[get_lat_lon, get_nws_alerts, get_nws_forecast],
    **CALLBACKS,
)

# The ADK built-in google_search cannot be combined with other tools in one agent,
# which is exactly why it lives here alone and is exposed via AgentTool.
search_agent = Agent(
    name="ReadyNow_Search",
    model=MODEL,
    description="Searches the internet for current disaster news, road closures, and open shelters.",
    instruction=(
        "Use google_search to find current, local emergency information for the request. "
        "Prefer official sources (.gov, NWS, county emergency management, American Red Cross). "
        "Report what you actually found and say plainly when you could not confirm something."
    ),
    tools=[google_search],
    **CALLBACKS,
)

routes_agent = Agent(
    name="ReadyNow_Routes",
    model=MODEL,
    description="Provides advisory evacuation routes using the Google Maps Directions API.",
    instruction=(
        "Run get_evacuation_route with the origin and destination. Always include the advisory "
        "that official evacuation orders take precedence over any route you provide."
    ),
    tools=[get_evacuation_route],
    **CALLBACKS,
)


# =============================================================================
# 4. SEQUENTIAL WORKFLOW — draft, validate, refine
# =============================================================================

responder = Agent(
    name="ReadyNow_Responder",
    model=MODEL,
    description="Answers the user's emergency question by calling the specialist agents.",
    instruction=(
        "You are ReadyNow!, FEMA's emergency preparedness assistant. Draft a briefing that "
        "answers the user's question.\n"
        "Call ReadyNow_Weather for alerts and forecasts, ReadyNow_Search for shelters, news, "
        "and closures, and ReadyNow_Routes for evacuation routing. Call more than one when the "
        "question needs it, but call them ONE AT A TIME - wait for each result before calling "
        "the next, to stay within model quota. You may answer general preparedness questions "
        "from your own knowledge.\n"
        "If the user describes an immediate threat to life, lead with 'Call 911'.\n"
        "Never invent shelter addresses, road closures, or alert text."
    ),
    tools=[
        AgentTool(agent=weather_agent),
        AgentTool(agent=search_agent),
        AgentTool(agent=routes_agent),
    ],
    output_key="draft_answer",
    **CALLBACKS,
)

critic = Agent(
    name="ReadyNow_Critic",
    model=MODEL,
    description="Audits the draft briefing for life-safety gaps and unsupported claims.",
    instruction=(
        "Audit this draft emergency briefing:\n\n{draft_answer}\n\n"
        "List concrete, numbered problems only. Check for: a failed tool lookup being presented "
        "as an all-clear; a location the user never provided; specifics (addresses, closures, "
        "alert text) that no tool returned; a missing life-safety advisory or 911 referral; "
        "and anything unclear to a frightened reader.\n"
        "Also flag RELEVANCE problems: content not scoped to the CURRENT emergency - "
        "year-round social services, unrelated crisis lines, or general directories padding "
        "out a briefing. In an emergency the reader must reach what they need fast, so say "
        "which items to cut or move below the immediate guidance.\n"
        "If the draft is sound, reply exactly: NO ISSUES."
    ),
    output_key="critique",
    **CALLBACKS,
)

refiner = Agent(
    name="ReadyNow_Refiner",
    model=MODEL,
    description="Rewrites the briefing applying the critique, and produces the final output.",
    instruction=(
        "Rewrite this draft:\n\n{draft_answer}\n\nApplying this critique:\n\n{critique}\n\n"
        "If the critique is 'NO ISSUES', return the draft with formatting cleaned up only. "
        "Output the final briefing directly to the user: bold section headers, short bullets, "
        "most urgent item first. Do not mention the draft, the critique, or this process."
    ),
    output_key="final_briefing",
    **CALLBACKS,
)

response_team = SequentialAgent(
    name="ReadyNow_ResponseTeam",
    description="Drafts an emergency briefing, validates it, then refines it before it reaches the user.",
    sub_agents=[responder, critic, refiner],
)


# =============================================================================
# 5. ROOT AGENT + AGENT ENGINE APP
# =============================================================================

readynow_root = Agent(
    name="ReadyNow_Root",
    model=MODEL,
    description="FEMA ReadyNow! emergency preparedness assistant.",
    instruction=(
        "You are ReadyNow!, FEMA's emergency preparedness assistant. You provide real-time "
        "weather alerts, disaster safety guidance, evacuation routing, and emergency shelter "
        "resources for U.S. locations.\n"
        "If the user only greets you or asks what you can do, answer directly and briefly.\n"
        "For any actual emergency, weather, shelter, safety, or routing request, transfer to "
        "ReadyNow_ResponseTeam, which drafts, validates, and refines the briefing."
    ),
    sub_agents=[response_team],
    **CALLBACKS,
)

root_agent = readynow_root  # ADK convention

app = agent_engines.AdkApp(agent=root_agent, enable_tracing=True)

print("Architecture built.")
print(f"  Project : {PROJECT_ID} ({LOCATION})")
print(f"  Staging : {STAGING_BUCKET}")
print(f"  Maps key: {'set' if os.environ.get('GOOGLE_MAPS_API_KEY') else 'NOT SET (degraded mode)'}")
print(f"  Root    : {root_agent.name} -> {response_team.name} -> "
      f"{[a.name for a in response_team.sub_agents]}")


Created staging bucket gs://qwiklabs-gcp-00-d39fe808a9e1-readynow-staging
Architecture built.
  Project : qwiklabs-gcp-00-d39fe808a9e1 (us-central1)
  Staging : gs://qwiklabs-gcp-00-d39fe808a9e1-readynow-staging
  Maps key: NOT SET (degraded mode)
  Root    : ReadyNow_Root -> ReadyNow_ResponseTeam -> ['ReadyNow_Responder', 'ReadyNow_Critic', 'ReadyNow_Refiner']


/tmp/ipykernel_68487/2729391561.py:463: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  response_team = SequentialAgent(


# Agent Local Testing

In [ ]:
# =============================================================================
# LOCAL TESTING
# =============================================================================


TESTS = [
    ("Weather + alerts", "Are there any active weather alerts for Bowie, Maryland?"),
    ("Multi-agent",      "A hurricane is coming to Tampa, Florida. What's the forecast, "
                         "where are the open shelters, and how do I drive to Orlando?"),
    ("Input validation", "Tell me a joke and give me a recipe for lasagna."),
    ("Non-US rejection", "What's the weather in London?"),
]


def _session_id(session):
    """create_session returns a dict locally and remotely in current SDK versions."""
    return session["id"] if isinstance(session, dict) else session.id


def run_local(label, message, show_events=False, retries=3):
    print("=" * 78)
    print(f"TEST: {label}\nUSER: {message}")
    print("-" * 78)
    for attempt in range(retries):
        session = app.create_session(user_id="local-test-user")
        final = None
        try:
            for event in app.stream_query(
                user_id="local-test-user", session_id=_session_id(session), message=message
            ):
                author = event.get("author", "?")
                for part in (event.get("content") or {}).get("parts", []) or []:
                    if show_events and part.get("function_call"):
                        print(f"   [{author}] CALL   {part['function_call'].get('name')}")
                    if show_events and part.get("function_response"):
                        print(f"   [{author}] RESULT {part['function_response'].get('name')}")
                    if part.get("text"):
                        print(f"   [{author}] {part['text'].strip()[:200]}")
                        final = part["text"]
        except Exception as exc:
            print(f"   [run] attempt {attempt + 1} raised: {type(exc).__name__}")
        if final:
            break
        # A 429 raises on ADK's background thread, so it reaches us as an empty
        # stream rather than an exception. Back off and try again.
        if attempt < retries - 1:
            wait = 20 * (attempt + 1)
            print(f"   [run] no output - likely quota (429); retrying in {wait}s")
            time.sleep(wait)
    print("-" * 78)
    print("FINAL:\n" + (final or "(no text returned after retries)"))
    print()
    return final


# Run one test with full event tracing to prove sub-agent and tool delegation.
run_local(TESTS[1][0], TESTS[1][1], show_events=True)

# Then the rest.
for label, message in [TESTS[0], TESTS[2], TESTS[3]]:
    run_local(label, message)

# Agent Runtime Deployment and Remote Testing

In [ ]:
# =============================================================================
# DEPLOY TO AGENT ENGINE + REMOTE TESTING
# =============================================================================
#

from importlib import metadata


def pinned_requirements():
    """Pin to the versions installed in THIS kernel.

    Version skew between the notebook and the Agent Engine container is the most
    common cause of a deploy that builds fine but fails at runtime.
    """
    pins = []
    for pkg in ("google-cloud-aiplatform", "google-adk", "requests"):
        try:
            pins.append(f"{pkg}=={metadata.version(pkg)}")
        except metadata.PackageNotFoundError:
            pins.append(pkg)
    pins[0] = pins[0].replace(
        "google-cloud-aiplatform==", "google-cloud-aiplatform[agent_engines,adk]=="
    )
    return pins


# Environment variables must be passed explicitly — os.environ in this notebook
# does NOT carry into the deployed container.
env_vars = {
    k: v for k, v in {
        "GOOGLE_MAPS_API_KEY": os.environ.get("GOOGLE_MAPS_API_KEY"),
        "NWS_USER_AGENT": os.environ.get("NWS_USER_AGENT"),
    }.items() if v
}

existing = next(
    (e for e in agent_engines.list() if getattr(e, "display_name", None) == DISPLAY_NAME),
    None,
)

if existing is not None:
    print(f"Updating existing engine: {existing.resource_name}")
    remote_app = existing.update(
        agent_engine=app,
        requirements=pinned_requirements(),
        display_name=DISPLAY_NAME,
        env_vars=env_vars or None,
    )
else:
    print("Creating new Agent Engine instance (5-10 min)...")
    remote_app = agent_engines.create(
        app,
        display_name=DISPLAY_NAME,
        description="FEMA ReadyNow! emergency preparedness assistant (Case Study 6).",
        requirements=pinned_requirements(),
        env_vars=env_vars or None,
    )

ENGINE_ID = remote_app.resource_name.split("/")[-1]
print(f"\nDeployed: {remote_app.resource_name}")


# ----------------------------------------------------------------------------
# Remote tests against the deployed endpoint
# ----------------------------------------------------------------------------
def run_remote(label, message):
    print("=" * 78)
    print(f"REMOTE TEST: {label}\nUSER: {message}")
    print("-" * 78)
    session = remote_app.create_session(user_id="remote-test-user")
    sid = session["id"] if isinstance(session, dict) else session.id
    final = None
    for event in remote_app.stream_query(
        user_id="remote-test-user", session_id=sid, message=message
    ):
        author = event.get("author", "?")
        for part in (event.get("content") or {}).get("parts", []) or []:
            if part.get("function_call"):
                print(f"   [{author}] CALL   {part['function_call'].get('name')}")
            if part.get("text"):
                final = part["text"]
    print("FINAL:\n" + (final or "(no text returned)"))
    print()


import time

run_remote("Deployed weather + alerts", "Are there any active weather alerts for Bowie, Maryland?")
time.sleep(20)  # stay under the per-minute model quota
run_remote("Deployed multi-agent", "Wildfire near Boulder, Colorado — where do I go and how do I get there?")
time.sleep(20)
run_remote("Deployed input validation", "Write me an essay about the French Revolution.")


# ----------------------------------------------------------------------------
# Agent Playground
# ----------------------------------------------------------------------------
print("=" * 78)
print("TEST IN THE AGENT PLAYGROUND")
print("=" * 78)
print(f"https://console.cloud.google.com/vertex-ai/agents/agent-engines?project={PROJECT_ID}")
print()
print("  1. Open the link (Vertex AI > Agent Engine).")
print(f"  2. Click the instance named: {DISPLAY_NAME}")
print("  3. Click the Playground tab, type a message, and use 'New Session' to start over.")
print("  4. The Trace tab shows the sub-agent and tool calls for each turn")
print("     (enable_tracing=True was set on the AdkApp) — good evidence for grading.")
print()
print(f"  Resource name : {remote_app.resource_name}")
print(f"  Engine ID     : {ENGINE_ID}")
print()
print("  Cleanup when finished:  remote_app.delete(force=True)")
